# Pudú Perceptual Shorts — Ep. 01 «La luz que acelera» (beep-speed illusion)

**Desde Android:** abre este notebook en Colab y ejecuta las celdas **en orden** con ▶.

1. `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU` (gratis, si hay disponibilidad).
2. Ejecuta las celdas 1 → 5. La celda 5 descarga el MP4 al teléfono (o lo copia a Google Drive).

Duración aproximada con GPU T4: prueba de 8 s ≈ 3–6 min; versión final ≈ 30–60 min (estimación, no medida en T4).
Sin GPU funciona igual en CPU, pero mucho más lento (el modo `animatic` tarda unos 10 min en CPU).

In [ ]:
#@title 1 · Clonar el proyecto
REPO = "https://github.com/ManIUs-X/VideosAI.git"  #@param {type:"string"}
BRANCH = "claude/pudu-perceptual-shorts-x7bxef"  #@param {type:"string"}
# Repositorio privado: guarda un token de GitHub en Colab → 🔑 Secretos con el nombre GITHUB_TOKEN
import os, subprocess
url = REPO
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
    if tok:
        url = REPO.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.exists("/content/videosai"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/content/videosai"], check=True)
%cd /content/videosai
!git log --oneline -1

In [ ]:
#@title 2 · Instalar Blender 4.2 LTS y dependencias (≈2 min)
!bash tools/install_blender.sh
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Sin GPU: se usará CPU" 

In [ ]:
#@title 3 · Configurar
MODE = "test"  #@param ["test", "animatic", "final", "final_hd"]
import shutil, subprocess
has_gpu = shutil.which("nvidia-smi") and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
DEVICE = "OPTIX" if has_gpu else "CPU"
print("modo:", MODE, "| dispositivo:", DEVICE)

In [ ]:
#@title 4 · Renderizar, montar y verificar (re-ejecutable: reanuda frames ya hechos)
!python run.py --episode ep01_beep_speed --mode {MODE} --device {DEVICE}

In [ ]:
#@title 5 · Ver y descargar el MP4
import glob, os
from IPython.display import HTML
from base64 import b64encode
out = f"output/ep01_beep_speed/{MODE}/ep01_beep_speed_{MODE}.mp4"
print(open(f"output/ep01_beep_speed/{MODE}/verification.json").read()[:1500])
if os.path.getsize(out) < 60e6:
    data = b64encode(open(out, "rb").read()).decode()
    display(HTML(f'<video width=270 controls src="data:video/mp4;base64,{data}"></video>'))
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/PuduShorts", exist_ok=True)
    for f in glob.glob(f"output/ep01_beep_speed/{MODE}/*.mp4") + glob.glob(f"output/ep01_beep_speed/{MODE}/verification.json"):
        shutil.copy(f, "/content/drive/MyDrive/PuduShorts/")
    print("Copiado a Drive/PuduShorts")
else:
    from google.colab import files
    files.download(out)